## Parte 2:Preprocesamiento de datos

1. Transformación de Columnas:
*   Utilizar ColumnTransformer para aplicar transformaciones específicas a diferentes columnas.
*   Realizar codificación de variables categóricas y escalado de variables numéricas.
2. Pipelines:
*   Crear pipelines para automatizar el preprocesamiento de datos y asegurar la reproducibilidad.

Como siempre, cargamos nuevamente el dataset y dividimos

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                            f1_score, classification_report, confusion_matrix, 
                            ConfusionMatrixDisplay, roc_curve, roc_auc_score)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer, OneHotEncoder, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# Ignorar advertencias
warnings.filterwarnings("ignore")

#Cargamos el dataset

df = pd.read_csv('../data/retail_sales_dataset.csv')

#Separamos las variables predictoras y la variable objetivo
X = df.drop(columns=['Gender'])
y = df['Gender']

#Separamos el dataset en conjunto de entrenamiento y conjunto de prueba

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Convertimos la columna 'Date' a tipo datetime y extraemos el mes

for dataset in [X_train, X_test]:
    dataset['Date'] = pd.to_datetime(dataset['Date'], errors='coerce')
    dataset['Mes'] = dataset['Date'].dt.month
    dataset.drop(columns=['Date'], inplace=True)


Codificar nuestras columnas y escalamos, definiendo Pipelines

In [10]:
categorical_features=['Product Category']
numerical_features=['Age', 'Mes', 'Quantity', 'Price per Unit']

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant',fill_value='other')), #creamos la categoría "other", para los datos faltantes
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

numerical_transformer = Pipeline(steps=[
    ('imputer_nan', KNNImputer(n_neighbors=5)), #imputamos con KNN los datos a los faltantes
    ('scaler',StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', categorical_transformer, categorical_features),
        ('numerical', numerical_transformer, numerical_features)
        ])

Codificamos nuestra variable objetivo, para poder realizar la clasificación

In [11]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)